# Lyrics Transcribe in Colab

**Pipeline:** anvuew BS-Roformer (vocal isolation) → WhisperX or GigaAM (ASR with word timestamps) → standalone karaoke HTML page.

**Runtime:** GPU required — *Runtime → Change runtime type → T4 GPU* (or better).

**What you upload:** only the audio file(s) you want to transcribe. The code is cloned from GitHub, the anvuew checkpoint is fetched from a release, WhisperX (~3 GB from HuggingFace) and GigaAM (~430 MB from Sber CDN) auto-download into `models/` on first use.

## 1. Clone the repo

In [ ]:
!git clone https://github.com/Beatloo-Labs/LyricsTranscribe.git /content/LyricsTranscribe
%cd /content/LyricsTranscribe
!ls -la

## 2. Create venv + install dependencies via uv

Note: `gigaam` is installed from git, not PyPI — the PyPI 0.1.0 wheel has stricter pins that downgrade torch and break faster-whisper. The git repo has loose `torch>=2.6` (optional extra).

In [ ]:
!pip install -q uv
!uv venv
# torch must come from the pytorch index for the CUDA build
!uv pip install -p .venv/bin/python --index-url https://download.pytorch.org/whl/cu128 torch==2.8.0 torchaudio==2.8.0
!uv pip install -p .venv/bin/python -r requirements.txt

## 3. Environment variables + sanity check

`LD_LIBRARY_PATH` needs the cuDNN libs torch installed (via the `nvidia-cudnn-cu*` pip wheel) plus the system NVIDIA driver libs. We resolve the cuDNN path with a glob since the Python minor version under `.venv/lib/` varies between Colab images.

In [ ]:
import os, glob

cudnn_dirs = glob.glob('/content/LyricsTranscribe/.venv/lib/python*/site-packages/nvidia/cudnn/lib/')
if not cudnn_dirs:
    raise RuntimeError('cuDNN libs not found — did the torch install succeed?')

os.environ['MPLBACKEND'] = 'agg'
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = 'true'
os.environ['LD_LIBRARY_PATH'] = (
    cudnn_dirs[0] + ':/usr/lib64-nvidia:' + os.environ.get('LD_LIBRARY_PATH', '')
)
print('cuDNN libs:', cudnn_dirs[0])
print('LD_LIBRARY_PATH:', os.environ['LD_LIBRARY_PATH'])

In [ ]:
# Sanity check: torch sees CUDA, gigaam imports, faster-whisper finds cuDNN.
import subprocess
subprocess.run(['.venv/bin/python', '-c', '''
import torch
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")
import faster_whisper, gigaam, silero_vad
print("faster_whisper, gigaam, silero_vad: OK")
'''], env=os.environ, check=True)

## 4. Download anvuew checkpoint (~196 MB)

Fetched from the project's GitHub release. The file exceeds GitHub's 100 MB per-file limit so it's hosted in a release rather than the repo.

In [ ]:
!mkdir -p models/anvuew
!wget -q --show-progress \
  https://github.com/Beatloo-Labs/LyricsTranscribe/releases/download/weights-v1/bs_roformer_ft1_anvuew_sdr_12.55.ckpt \
  -O models/anvuew/bs_roformer_ft1_anvuew_sdr_12.55.ckpt
!ls -lh models/anvuew/

## 5. Upload audio file(s) to transcribe

In [ ]:
from google.colab import files

# 1) Upload audio file(s) — mp3 / wav / flac / m4a / ogg
print('Upload AUDIO file(s):')
audio = files.upload()
AUDIO_FILES = [n for n in audio if not n.lower().endswith('.txt')]
print('audio:', AUDIO_FILES)

# 2) (optional) upload a .txt with the exact lyrics. If you upload one,
#    the next cell will switch to forced-alignment mode automatically:
#    words come from your text, timings from MMS — no ASR errors.
#    Format: one phrase per line, UTF-8. Skip this to run plain ASR.
print('\nUpload lyrics .txt (optional, cancel to skip):')
try:
    lyrics = files.upload()
    LYRICS_FILE = next((n for n in lyrics if n.lower().endswith('.txt')), None)
except Exception:
    LYRICS_FILE = None
print('lyrics:', LYRICS_FILE or '(none — will use ASR)')


## 6. Run the CLI

Pick `--model whisperx` (multilingual) or `--model gigaam` (Russian-tuned). First run downloads the chosen ASR model (~3 GB for whisperx, ~430 MB for gigaam) into `models/`. Each track lands in its own `cli-out/<name>__<model>__<timestamp>/` subfolder with the audio, JSON, and karaoke HTML.

In [ ]:
import subprocess

MODEL    = 'whisperx'   # 'whisperx' or 'gigaam' (ignored if LYRICS_FILE is set)
LANGUAGE = 'ru'         # ISO code, or empty string for auto-detect
ISOLATE  = True         # run anvuew vocal separation before ASR/align
OUT_DIR  = 'cli-out'

args = ['.venv/bin/python', 'transcribe.py',
        '--language', LANGUAGE, '--out', OUT_DIR]
if not ISOLATE:
    args.append('--no-isolate')

if LYRICS_FILE:
    # forced alignment: words from your .txt, timings from MMS
    # --model is ignored in this mode
    args += ['--lyrics', LYRICS_FILE]
    print(f'[align] using lyrics: {LYRICS_FILE}')
else:
    args += ['--model', MODEL]
    print(f'[asr] using model: {MODEL}')

args += AUDIO_FILES
subprocess.run(args, env=os.environ, check=True)


## 7. Download results

Each track produces three files in its `cli-out/<name>__<model>__<timestamp>/` subfolder: the original audio, `<name>.json` (microformat), and `<name>.html` (standalone karaoke page that auto-loads the sibling audio). Zip everything and download:

In [ ]:
!ls -la cli-out/
!cd cli-out && zip -rq ../karaoke-output.zip .
from google.colab import files
files.download('karaoke-output.zip')

## (Optional) Run the FastAPI server in Colab

Colab can't expose a port to the public web directly, but `serve_kernel_port_as_window` proxies it through the Colab UI.

In [ ]:
import subprocess, time
proc = subprocess.Popen(['.venv/bin/python', 'server.py'], env=os.environ)
time.sleep(4)
from google.colab.output import serve_kernel_port_as_window
serve_kernel_port_as_window(8000)